# The Phase Vocoder

The phase vocoder analyzes sound in overlapping FFT frames and resynthesizes from the evolving magnitude/phase spectrum -- which lets it change **pitch without changing duration** (and vice versa, combined with `src`).

Source: [CLM generators -- phase-vocoder](https://ccrma.stanford.edu/software/snd/snd/sndclm.html)

In [ ]:
import os, importlib
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

import pysndlib.clm as clm

SR = 44100

examples_dir = os.path.dirname(importlib.import_module('pysndlib.examples').__file__)
EXAMPLE = os.path.join(examples_dir, 'flute_trill_1.wav')

def pitch_shift(src_file, out, pitch, dur):
    with clm.Sound(out, channels=1, srate=SR):
        reader = clm.make_readin(src_file)
        pv = clm.make_phase_vocoder(reader, fft_size=512, overlap=4, interp=128, pitch=pitch)
        for i in range(int(dur * SR)):
            clm.outa(i, clm.phase_vocoder(pv))
    d, _ = clm.file2ndarray(out)
    return d[0]

def peak_hz(x, n=8192):
    rl = np.copy(x[:n].astype(float)); im = np.zeros(n)
    clm.mus_fft(rl, im, n, 1); clm.mus_rectangular2polar(rl, im)
    return np.argmax(np.abs(rl[:n//2])) * SR / n

## Anatomy

`make_phase_vocoder(input, fft_size, overlap, interp, pitch)`:
- `fft_size` / `overlap`: analysis window and hop (fft_size/overlap = hop)
- `pitch`: the pitch ratio, applied **with time preserved** -- unlike tape speed

## Pitch shifting with time preserved

In [ ]:
data, _ = clm.file2ndarray(EXAMPLE)
print('input duration :', data.shape[1] / SR, 's')
ipd.Audio(EXAMPLE, rate=SR)

In [ ]:
up = pitch_shift(EXAMPLE, 'pv_up.wav', 2.0, 2.0)
down = pitch_shift(EXAMPLE, 'pv_down.wav', 0.5, 2.0)
print('up   : duration %.2f s, peak %.0f Hz' % (len(up)/SR, peak_hz(up[SR//10:])))
print('down : duration %.2f s, peak %.0f Hz' % (len(down)/SR, peak_hz(down[SR//10:])))

In [ ]:
ipd.Audio('pv_up.wav', rate=SR)

In [ ]:
ipd.Audio('pv_down.wav', rate=SR)

In [ ]:
plt.figure(figsize=(14, 4))
plt.subplot(1, 2, 1)
plt.specgram(up[:SR*2], NFFT=2048, Fs=SR, noverlap=512); plt.title('pitch 2.0 (octave up)')
plt.subplot(1, 2, 2)
plt.specgram(down[:SR*2], NFFT=2048, Fs=SR, noverlap=512); plt.title('pitch 0.5 (octave down)')
plt.show()

## Frame size tradeoffs

Bigger frames resolve frequency better but smear transients (and vice versa). Compare 512 vs 4096 point analysis:

In [ ]:
small = pitch_shift(EXAMPLE, 'pv_small.wav', 2.0, 1.0)
with clm.Sound('pv_big.wav', channels=1, srate=SR):
    reader = clm.make_readin(EXAMPLE)
    pv = clm.make_phase_vocoder(reader, fft_size=4096, overlap=4, interp=128, pitch=2.0)
    for i in range(SR):
        clm.outa(i, clm.phase_vocoder(pv))
big, _ = clm.file2ndarray('pv_big.wav')

plt.figure(figsize=(14, 3))
plt.plot(small[:3000], lw=0.8, label='fft_size=512')
plt.plot(big[0][:3000], lw=0.8, alpha=0.7, label='fft_size=4096')
plt.legend(); plt.title('frame size: time resolution vs frequency resolution')
plt.show()

## The time-stretch recipe

`src` changes speed (pitch and duration together, tape-style). Combine it with the vocoder's pitch shift and the two changes cancel -- leaving pure time-stretching:

In [ ]:
# stretch to 2x duration: first pitch up an octave (time kept),
# then src at half speed (duration doubles, pitch drops back)
with clm.Sound('pv_stretched.wav', channels=1, srate=SR):
    reader = clm.make_readin('pv_up.wav')            # pv_up: flute +1 octave, same time
    s = clm.make_src(reader, srate=0.5)
    for i in range(2 * SR):
        clm.outa(i, clm.src(s, 0.0))

st, _ = clm.file2ndarray('pv_stretched.wav')
print('stretched duration: %.2f s' % (st.shape[1] / SR))

In [ ]:
ipd.Audio('pv_stretched.wav', rate=SR)

## Three ways to bend time

| tool | pitch | duration | character |
|------|-------|-----------|-----------|
| `src` | changes | changes | tape-like, clean |
| `granulate` | preserved | changes (expansion) | grainy, textural |
| `phase_vocoder` | changes | preserved (and recipes for the rest) | smooth, "vocoded" |

In [ ]:
ipd.Audio('pv_stretched.wav', rate=SR)

## Summary

- The phase vocoder decouples pitch from time
- Frame size trades time vs frequency resolution
- pv + src recipes give pitch-preserving time-stretch

Next: [spectral analysis tools](14_Spectral_Tools.ipynb)